In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import sys

# Works whether Jupyter starts from repo root or notebooks/
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "datasphere").exists() else cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATASETS_RAW = REPO_ROOT / "datasets" / "raw"
DATASETS_PROCESSED = REPO_ROOT / "datasets" / "processed"
MODELS_DIR = REPO_ROOT / "models"
for folder in (DATASETS_RAW, DATASETS_PROCESSED, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)


def numeric_only_columns(frame, candidates=None):
    """Return true numeric columns, excluding booleans."""
    cols = list(candidates) if candidates is not None else list(frame.columns)
    numeric = []
    for col in cols:
        if col not in frame.columns:
            continue
        if pd.api.types.is_bool_dtype(frame[col]):
            continue
        if pd.api.types.is_numeric_dtype(frame[col]):
            numeric.append(col)
    return numeric


def print_iqr_outliers(frame, columns):
    for col in numeric_only_columns(frame, columns):
        series = pd.to_numeric(frame[col], errors="coerce")
        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = frame[(series < lower_bound) | (series > upper_bound)]
        print(f"\n{col}")
        print("Lower Bound:", lower_bound)
        print("Upper Bound:", upper_bound)
        print("Number of Outliers:", len(outliers))

In [ ]:
try:
    from datasphere.data.loader import load_primary_dataset
    df = load_primary_dataset()
except FileNotFoundError:
    import pandas as pd
    from datasphere.data.archives import extract_dataset_archives
    extract_dataset_archives()
    csv_path = DATASETS_RAW / "child_education_risk_intelligence.csv"
    if not csv_path.exists():
        raise FileNotFoundError(
            "Dataset missing. Run: python -m pip install -e .[dev] and ensure "
            "datasets/raw/compressed_child_education_risk_intelligence.zip exists."
        )
    df = pd.read_csv(csv_path)

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")


In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape


In [ ]:
df.info()

In [ ]:
#missing value sum
df.isnull().sum() 


In [ ]:
df.isnull().sum()/df.shape[0]*100#percentage of missing values

In [ ]:
#finding duplicates
df.duplicated().sum()


In [ ]:
#identifying the garbage value
for i in df.select_dtypes(include="object").columns:
    print(df[i].value_counts())
    print("***"*10)


In [ ]:
#Exploratory Data Analysis(EDA)

In [ ]:
#descriptive statistics
df.describe().T


In [ ]:
df.describe(include="object")

In [ ]:
#histogram to understand the distribution
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

for i in numeric_only_columns(df):
    sns.histplot(data = df, x=i)
    plt.show()
    

In [ ]:
#boxplot-to-identify outliers
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

for i in numeric_only_columns(df):
    sns.boxplot(data = df, x=i)
    plt.show()

In [ ]:
#scatter plot to understand the relationship
for i in ['age', 'grade_level', 'distance_to_school_km',
       'household_income_monthly_usd', 'family_size', 'number_of_siblings',
       'attendance_rate_pct', 'average_test_score_pct',
       'number_of_school_transfers', 'disciplinary_incidents_count',
       'teacher_student_ratio', 'school_infrastructure_score']:
    sns.scatterplot(data=df, x=i, y='dropout_probability_score')
    plt.show()


In [ ]:
numeric_only_columns(df)

In [ ]:
#correlation with heatmap to interpret the relation and multicolliniarity
s = df[numeric_only_columns(df)].corr()

In [ ]:
plt.figure(figsize = (15,15))
sns.heatmap(s,annot=True)


In [ ]:
#miising value treatment


In [ ]:
#choose the method to input the missing value
#like mean,median, mode or KNNIputer


In [ ]:
for i in ["distance_to_school_km","household_income_monthly_usd","family_size","attendance_rate_pct",
"average_test_score_pct","teacher_student_ratio","school_infrastructure_score"]:
    df[i].fillna(df[i].median(),inplace=True)


In [ ]:
df.isnull().sum()

In [ ]:
df_clean = df.copy()


In [ ]:
numerical_cols = [
    'distance_to_school_km',
    'household_income_monthly_usd',
    'family_size',
    'attendance_rate_pct',
    'average_test_score_pct',
    'teacher_student_ratio',
    'school_infrastructure_score'
]


In [ ]:
df_clean[numerical_cols].median()

In [ ]:
for col in numeric_only_columns(df_clean, numerical_cols):
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

In [ ]:
df_clean[numerical_cols].isnull().sum()


In [ ]:
categorical_cols = [
    'mother_education_level',
    'father_education_level',
    'household_has_internet_access'
]


In [ ]:
df_clean['mother_education_level'].mode()[0]

In [ ]:
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])


In [ ]:
df_clean['mother_education_level'] = (
    df_clean['mother_education_level'].fillna('Unknown')
)

df_clean['father_education_level'] = (
    df_clean['father_education_level'].fillna('Unknown')
)


In [ ]:
df_clean['household_has_internet_access'] = (
    df_clean['household_has_internet_access'].fillna('Unknown')
)


In [ ]:
df_clean['case_notes'] = df_clean['case_notes'].fillna('No notes')

In [ ]:
df_clean = df_clean.drop(columns=['case_notes'])

In [ ]:
df_clean.isnull().sum()

In [ ]:
df_clean['attendance_rate_pct'] = (
    df_clean['attendance_rate_pct']
    .fillna(df_clean['attendance_rate_pct'].median())
)


In [ ]:
df['attendance_rate_pct'].describe()

In [ ]:
df_clean['attendance_rate_pct'].describe()

In [ ]:
from pathlib import Path

cleaned_path = DATASETS_PROCESSED / "child_education_risk_intelligence_cleaned.csv"
cleaned_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(cleaned_path, index=False)
print(f"Cleaned dataset saved to {cleaned_path.resolve()}")


In [ ]:
print("Cleaned dataset saved successfully!")

In [ ]:
df['attendance_rate_pct'].skew()

In [ ]:
df['attendance_rate_pct'].plot(
    kind='hist',
    bins=30,
    figsize=(8, 5)
)


In [ ]:
print("Mean:", df['attendance_rate_pct'].mean())
print("Median:", df['attendance_rate_pct'].median())
print("Skewness:", df['attendance_rate_pct'].skew())


In [ ]:
df['attendance_rate_pct'] = df['attendance_rate_pct'].fillna(
    df['attendance_rate_pct'].median()
)


In [ ]:
df_clean.isnull().sum()

In [ ]:
#outliers treatment

In [ ]:
numerical_cols = [
    "age",
    "grade_level",
    "distance_to_school_km",
    "household_income_monthly_usd",
    "family_size",
    "number_of_siblings",
    "attendance_rate_pct",
    "average_test_score_pct",
    "number_of_school_transfers",
    "disciplinary_incidents_count",
    "teacher_student_ratio",
    "school_infrastructure_score",
    "dropout_probability_score",
]


In [ ]:
print_iqr_outliers(df, numerical_cols)


In [ ]:
numerical_cols = numeric_only_columns(df)
print(numerical_cols)


In [ ]:
continuous_cols = [
    "distance_to_school_km",
    "household_income_monthly_usd",
    "attendance_rate_pct",
    "average_test_score_pct",
    "teacher_student_ratio",
    "school_infrastructure_score"
]


In [ ]:
print_iqr_outliers(df, continuous_cols)


In [ ]:
print(numerical_cols)

In [ ]:
print_iqr_outliers(df, numerical_cols)


In [ ]:
col = "distance_to_school_km"

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df[
    (df[col] < lower_bound) |
    (df[col] > upper_bound)
][[col]].sort_values(by=col)


In [ ]:
col = "distance_to_school_km"

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df[
    (df[col] < lower_bound) |
    (df[col] > upper_bound)
][[col]].sort_values(by=col)


In [ ]:
import matplotlib.pyplot as plt

for col in numeric_only_columns(df, numerical_cols):

    plt.figure(figsize=(8, 4))

    plt.boxplot(df[col].dropna())

    plt.title(f"Boxplot - {col}")
    plt.ylabel(col)

    plt.show()

In [ ]:
continuous_cols = [
    "distance_to_school_km",
    "household_income_monthly_usd",
    "attendance_rate_pct",
    "average_test_score_pct",
    "teacher_student_ratio",
    "school_infrastructure_score"
]


In [ ]:
count_cols = [
    "number_of_siblings",
    "number_of_school_transfers",
    "disciplinary_incidents_count",
    "bullying_incidents_reported",
    "health_issues_reported"
]


In [ ]:
discrete_cols = [
    "age",
    "grade_level"
]


In [ ]:
df[
    ~df["attendance_rate_pct"].between(0, 100)
]


In [ ]:
df[
    ~df["average_test_score_pct"].between(0, 100)
]


In [ ]:
print(
    "Invalid attendance:",
    (~df["attendance_rate_pct"].between(0, 100)).sum()
)

print(
    "Invalid test score:",
    (~df["average_test_score_pct"].between(0, 100)).sum()
)

print(
    "Invalid infrastructure score:",
    (~df["school_infrastructure_score"].between(0, 100)).sum()
)

print(
    "Invalid dropout probability:",
    (~df["dropout_probability_score"].between(0, 1)).sum()
)

print(
    "Invalid age:",
    (~df["age"].between(5, 19)).sum()
)

print(
    "Invalid grade:",
    (~df["grade_level"].between(1, 12)).sum()
)


In [ ]:
df[
    ~df["average_test_score_pct"].between(0, 100)
]


In [ ]:
~df["average_test_score_pct"].between(0, 100)

In [ ]:
print("Missing test scores:",
      df["average_test_score_pct"].isna().sum())

print("Missing infrastructure:",
      df["school_infrastructure_score"].isna().sum())


In [ ]:
invalid_test_score = df[
    df["average_test_score_pct"].notna() &
    ~df["average_test_score_pct"].between(0, 100)
]

print("Invalid test scores:", len(invalid_test_score))


In [ ]:
invalid_infrastructure = df[
    df["school_infrastructure_score"].notna() &
    ~df["school_infrastructure_score"].between(0, 100)
]

print("Invalid infrastructure scores:",
      len(invalid_infrastructure))


In [ ]:
print(
    "Invalid attendance:",
    (
        df["attendance_rate_pct"].notna() &
        ~df["attendance_rate_pct"].between(0, 100)
    ).sum()
)

print(
    "Invalid test score:",
    (
        df["average_test_score_pct"].notna() &
        ~df["average_test_score_pct"].between(0, 100)
    ).sum()
)

print(
    "Invalid infrastructure score:",
    (
        df["school_infrastructure_score"].notna() &
        ~df["school_infrastructure_score"].between(0, 100)
    ).sum()
)

print(
    "Invalid dropout probability:",
    (
        df["dropout_probability_score"].notna() &
        ~df["dropout_probability_score"].between(0, 1)
    ).sum()
)

print(
    "Invalid age:",
    (
        df["age"].notna() &
        ~df["age"].between(5, 19)
    ).sum()
)

print(
    "Invalid grade:",
    (
        df["grade_level"].notna() &
        ~df["grade_level"].between(1, 12)
    ).sum()
)


In [ ]:
for i in [
    "distance_to_school_km",
    "household_income_monthly_usd",
    "family_size",
    "attendance_rate_pct",
    "average_test_score_pct",
    "teacher_student_ratio",
    "school_infrastructure_score"
]:
    df[i] = df[i].fillna(df[i].median())


In [ ]:
df["average_test_score_pct"].isna().sum()

In [ ]:
df["school_infrastructure_score"].isna().sum()

In [ ]:
print(df["average_test_score_pct"].isna().sum())
print(df["school_infrastructure_score"].isna().sum())


In [ ]:
for col in [
    "distance_to_school_km",
    "household_income_monthly_usd",
    "family_size",
    "attendance_rate_pct",
    "average_test_score_pct",
    "teacher_student_ratio",
    "school_infrastructure_score"
]:
    df[col] = df[col].fillna(df[col].median())


In [ ]:
print(df["average_test_score_pct"].isna().sum())
print(df["school_infrastructure_score"].isna().sum())


In [ ]:
print(df["average_test_score_pct"].describe())
print(df["school_infrastructure_score"].describe())


In [ ]:
print(
    df[
        ~df["average_test_score_pct"].between(0, 100)
    ]["average_test_score_pct"]
)


In [ ]:
print(
    df[
        ~df["school_infrastructure_score"].between(0, 100)
    ]["school_infrastructure_score"]
)


In [ ]:
~df["average_test_score_pct"].between(0, 100)

In [ ]:
df["average_test_score_pct"] = df["average_test_score_pct"].fillna(
    df["average_test_score_pct"].median()
)


In [ ]:
df["average_test_score_pct"].isnull().sum()

In [ ]:
(~df["average_test_score_pct"].between(0, 100)).sum()

In [ ]:
(~df["school_infrastructure_score"].between(0, 100)).sum()

In [ ]:
(~df["attendance_rate_pct"].between(0, 100)).sum()

In [ ]:
~df["average_test_score_pct"].between(0, 100)

In [ ]:
continuous_cols = [
    "distance_to_school_km",
    "household_income_monthly_usd",
    "attendance_rate_pct",
    "average_test_score_pct",
    "teacher_student_ratio",
    "school_infrastructure_score"
]


In [ ]:
print_iqr_outliers(df, continuous_cols)


In [ ]:
Q1 = df["average_test_score_pct"].quantile(0.25)
Q3 = df["average_test_score_pct"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)


In [ ]:
Q1 = df["school_infrastructure_score"].quantile(0.25)
Q3 = df["school_infrastructure_score"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)


In [ ]:
for col in [
    "average_test_score_pct",
    "school_infrastructure_score"
]:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ]

    print("\n", col)
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Outlier count:", len(outliers))
    print("Outlier percentage:",
          len(outliers) / len(df) * 100)


In [ ]:
col = "average_test_score_pct"

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df[
    (df[col] < lower) |
    (df[col] > upper)
][col].value_counts().sort_index()


In [ ]:
col = "school_infrastructure_score"

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df[
    (df[col] < lower) |
    (df[col] > upper)
][col].value_counts().sort_index()


In [ ]:
Q1 = df["average_test_score_pct"].quantile(0.25)
Q3 = df["average_test_score_pct"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)


In [ ]:
outliers = df[
    (df["average_test_score_pct"] < lower_bound) |
    (df["average_test_score_pct"] > upper_bound)
]

print("Number of outliers:", len(outliers))
print("Percentage of outliers:", len(outliers) / len(df) * 100)


In [ ]:
print(
    outliers["average_test_score_pct"]
    .value_counts()
    .sort_index()
)


In [ ]:
lower_outliers = df[
    df["average_test_score_pct"] < lower_bound
]

upper_outliers = df[
    df["average_test_score_pct"] > upper_bound
]

print("Lower-side outliers:", len(lower_outliers))
print("Upper-side outliers:", len(upper_outliers))

print("\nLower-side values:")
print(
    lower_outliers["average_test_score_pct"]
    .value_counts()
    .sort_index()
)

print("\nUpper-side values:")
print(
    upper_outliers["average_test_score_pct"]
    .value_counts()
    .sort_index()
)


In [ ]:
Q1 = df["school_infrastructure_score"].quantile(0.25)
Q3 = df["school_infrastructure_score"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

lower_outliers = df[
    df["school_infrastructure_score"] < lower_bound
]

upper_outliers = df[
    df["school_infrastructure_score"] > upper_bound
]

print("\nLower-side outliers:", len(lower_outliers))
print("Upper-side outliers:", len(upper_outliers))


In [ ]:
print("\nLower-side values:")
print(
    lower_outliers["school_infrastructure_score"]
    .value_counts()
    .sort_index()
)

print("\nUpper-side values:")
print(
    upper_outliers["school_infrastructure_score"]
    .value_counts()
    .sort_index()
)


In [ ]:
Q1 = df["distance_to_school_km"].quantile(0.25)
Q3 = df["distance_to_school_km"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

lower_outliers = df[
    df["distance_to_school_km"] < lower_bound
]

upper_outliers = df[
    df["distance_to_school_km"] > upper_bound
]

print("\nLower-side outliers:", len(lower_outliers))
print("Upper-side outliers:", len(upper_outliers))


In [ ]:
print("\nLower-side values:")
print(
    lower_outliers["distance_to_school_km"]
    .value_counts()
    .sort_index()
)

print("\nUpper-side values:")
print(
    upper_outliers["distance_to_school_km"]
    .value_counts()
    .sort_index()
)


In [ ]:
print("Minimum:", df["distance_to_school_km"].min())
print("Maximum:", df["distance_to_school_km"].max())


In [ ]:
print(df["distance_to_school_km"].describe())

In [ ]:
Q1 = df["household_income_monthly_usd"].quantile(0.25)
Q3 = df["household_income_monthly_usd"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

lower_outliers = df[
    df["household_income_monthly_usd"] < lower_bound
]

upper_outliers = df[
    df["household_income_monthly_usd"] > upper_bound
]

print("\nLower-side outliers:", len(lower_outliers))
print("Upper-side outliers:", len(upper_outliers))


In [ ]:
print("\nLower-side values:")
print(
    lower_outliers["household_income_monthly_usd"]
    .value_counts()
    .sort_index()
)

print("\nUpper-side values:")
print(
    upper_outliers["household_income_monthly_usd"]
    .value_counts()
    .sort_index()
)


In [ ]:
print(
    df[df["distance_to_school_km"] < 0]["distance_to_school_km"]
)


In [ ]:
Q1 = df["household_income_monthly_usd"].quantile(0.25)
Q3 = df["household_income_monthly_usd"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

lower_outliers = df[
    df["household_income_monthly_usd"] < lower_bound
]

upper_outliers = df[
    df["household_income_monthly_usd"] > upper_bound
]

print("\nLower-side outliers:", len(lower_outliers))
print("Upper-side outliers:", len(upper_outliers))


In [ ]:
print("\nLower-side values:")
print(
    lower_outliers["household_income_monthly_usd"]
    .value_counts()
    .sort_index()
)

print("\nUpper-side values:")
print(
    upper_outliers["household_income_monthly_usd"]
    .value_counts()
    .sort_index()
)


In [ ]:
print(
    "Negative income values:",
    (df["household_income_monthly_usd"] < 0).sum()
)


In [ ]:
Q1 = df["household_income_monthly_usd"].quantile(0.25)
Q3 = df["household_income_monthly_usd"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

lower_outliers = df[
    df["household_income_monthly_usd"] < lower_bound
]

upper_outliers = df[
    df["household_income_monthly_usd"] > upper_bound
]

print("Lower-side outliers:", len(lower_outliers))
print("Upper-side outliers:", len(upper_outliers))


In [ ]:
print(df["household_income_monthly_usd"].describe())

In [ ]:
Q1 = df["household_income_monthly_usd"].quantile(0.25)
Q3 = df["household_income_monthly_usd"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

lower_outliers = df[
    df["household_income_monthly_usd"] < lower_bound
]

upper_outliers = df[
    df["household_income_monthly_usd"] > upper_bound
]

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

print("Lower-side outliers:", len(lower_outliers))
print("Upper-side outliers:", len(upper_outliers))


In [ ]:
print(
    upper_outliers["household_income_monthly_usd"]
    .describe()
)


In [ ]:
print(
    upper_outliers["household_income_monthly_usd"]
    .sort_values()
    .head(20)
)

print(
    upper_outliers["household_income_monthly_usd"]
    .sort_values()
    .tail(20)
)


In [ ]:
import numpy as np

df["household_income_monthly_usd_log"] = np.log1p(
    df["household_income_monthly_usd"]
)


In [ ]:
df["household_income_log"] = df["household_income_monthly_usd_log"]


In [ ]:
upper_outliers["household_income_monthly_usd"].describe()

In [ ]:
print(
    upper_outliers["household_income_monthly_usd"]
    .sort_values()
    .tail(20)
)


In [ ]:
print(
    upper_outliers["household_income_monthly_usd"]
    .sort_values()
    .head(20)
)


In [ ]:
import numpy as np

df["household_income_log"] = np.log1p(
    df["household_income_monthly_usd"]
)


In [ ]:
print(df["household_income_log"].describe())

In [ ]:
Q1 = df["attendance_rate_pct"].quantile(0.25)
Q3 = df["attendance_rate_pct"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

lower_outliers = df[
    df["attendance_rate_pct"] < lower_bound
]

upper_outliers = df[
    df["attendance_rate_pct"] > upper_bound
]

print("Lower-side outliers:", len(lower_outliers))
print("Upper-side outliers:", len(upper_outliers))


In [ ]:
print(df["attendance_rate_pct"].describe())

In [ ]:
print(
    "Invalid attendance values:",
    (~df["attendance_rate_pct"].between(0, 100)).sum()
)


In [ ]:
Q1 = df["attendance_rate_pct"].quantile(0.25)
Q3 = df["attendance_rate_pct"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

lower_outliers = df[
    df["attendance_rate_pct"] < lower_bound
]

upper_outliers = df[
    df["attendance_rate_pct"] > upper_bound
]

print("Lower-side outliers:", len(lower_outliers))
print("Upper-side outliers:", len(upper_outliers))


In [ ]:
print(df["attendance_rate_pct"].describe())

In [ ]:
print("\nLower-side values:")
print(
    lower_outliers["attendance_rate_pct"]
    .value_counts()
    .sort_index()
)

print("\nUpper-side values:")
print(
    upper_outliers["attendance_rate_pct"]
    .value_counts()
    .sort_index()
)


In [ ]:
# Don't do this
df["attendance_rate_pct"] = df["attendance_rate_pct"].clip(
    lower_bound,
    upper_bound
)


In [ ]:
Q1 = df["teacher_student_ratio"].quantile(0.25)
Q3 = df["teacher_student_ratio"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

lower_outliers = df[
    df["teacher_student_ratio"] < lower_bound
]

upper_outliers = df[
    df["teacher_student_ratio"] > upper_bound
]

print("Lower-side outliers:", len(lower_outliers))
print("Upper-side outliers:", len(upper_outliers))


In [ ]:
print(df["teacher_student_ratio"].describe())

In [ ]:
print("\nLower-side values:")
print(
    lower_outliers["teacher_student_ratio"]
    .value_counts()
    .sort_index()
)

print("\nUpper-side values:")
print(
    upper_outliers["teacher_student_ratio"]
    .value_counts()
    .sort_index()
)


In [ ]:
print(df["teacher_student_ratio"].describe())

In [ ]:
print(
    "Values <= 0:",
    (df["teacher_student_ratio"] <= 0).sum()
)

print(
    "Values > 100:",
    (df["teacher_student_ratio"] > 100).sum()
)


In [ ]:
print(
    "Minimum:",
    df["teacher_student_ratio"].min()
)

print(
    "Maximum:",
    df["teacher_student_ratio"].max()
)


In [ ]:
print(df["teacher_student_ratio"].describe())

In [ ]:
print("Ratio > 40:", (df["teacher_student_ratio"] > 40).sum())
print("Ratio > 50:", (df["teacher_student_ratio"] > 50).sum())
print("Ratio > 60:", (df["teacher_student_ratio"] > 60).sum())
print("Ratio > 70:", (df["teacher_student_ratio"] > 70).sum())
print("Ratio > 80:", (df["teacher_student_ratio"] > 80).sum())


In [ ]:
df["teacher_student_ratio"].clip(upper=60)

In [ ]:
print(df["family_size"].describe())

In [ ]:
print(
    "Family size <= 0:",
    (df["family_size"] <= 0).sum()
)


In [ ]:
print(
    df["family_size"]
    .value_counts()
    .sort_index()
)


In [ ]:
print("Family size <= 0:",
      (df["family_size"] <= 0).sum())

print("Minimum:",
      df["family_size"].min())

print("Maximum:",
      df["family_size"].max())


In [ ]:
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
df["mother_education_level"] = df["mother_education_level"].fillna("Unknown")

df["father_education_level"] = df["father_education_level"].fillna("Unknown")


In [ ]:
df["household_has_internet_access"] = (
    df["household_has_internet_access"].fillna("Unknown")
)


In [ ]:
df["case_notes"] = df["case_notes"].fillna("No information")

In [ ]:
df["mother_education_level"] = df["mother_education_level"].fillna("Unknown")

df["father_education_level"] = df["father_education_level"].fillna("Unknown")

df["household_has_internet_access"] = (
    df["household_has_internet_access"].fillna("Unknown")
)

df["case_notes"] = df["case_notes"].fillna("No information")


In [ ]:
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0]

In [ ]:
#duplicate records

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print("Duplicate student IDs:", df["student_id"].duplicated().sum())

In [ ]:
print("Unique student IDs:", df["student_id"].nunique())
print("Total rows:", len(df))


In [ ]:
print(df.dtypes)

In [ ]:
print(df.info())

In [ ]:
print(df["record_generated_date"].dtype)
print(df["enrollment_date"].dtype)


In [ ]:
print(df.dtypes)

In [ ]:
df["record_generated_date"] = pd.to_datetime(
    df["record_generated_date"],
    errors="coerce"
)

df["enrollment_date"] = pd.to_datetime(
    df["enrollment_date"],
    errors="coerce"
)


In [ ]:
print(df[["record_generated_date", "enrollment_date"]].dtypes)

In [ ]:
print(df[["record_generated_date", "enrollment_date"]].isnull().sum())

In [ ]:
print(
    df["household_has_internet_access"].value_counts(dropna=False)
)


In [ ]:
cols = [
    c for c in [
        "household_income_monthly_usd",
        "household_income_monthly_usd_log",
        "household_income_log",
    ]
    if c in df.columns
]
print(df[cols].head())


In [ ]:
if {"household_income_monthly_usd_log", "household_income_log"}.issubset(df.columns):
    print((df["household_income_monthly_usd_log"] == df["household_income_log"]).all())
else:
    print("Log columns already consolidated or not created yet.")


In [ ]:
if "household_income_monthly_usd_log" in df.columns and "household_income_log" in df.columns:
    df.drop(columns=["household_income_monthly_usd_log"], inplace=True)
    print("Dropped duplicate log column: household_income_monthly_usd_log")
else:
    print("Nothing to drop for income log columns.")


In [ ]:
import pandas as pd

df["record_generated_date"] = pd.to_datetime(
    df["record_generated_date"],
    errors="coerce"
)

df["enrollment_date"] = pd.to_datetime(
    df["enrollment_date"],
    errors="coerce"
)

print(df[["record_generated_date", "enrollment_date"]].dtypes)

print(
    df[["record_generated_date", "enrollment_date"]].isnull().sum()
)


In [ ]:
print(
    df["household_has_internet_access"].value_counts(dropna=False)
)


In [ ]:
pass  # duplicate income-log step removed; handled above


In [ ]:
print(df["dropout_risk_level"].value_counts(dropna=False))

print(df["dropout_probability_score"].describe())


In [ ]:
import pandas as pd

df["record_generated_date"] = pd.to_datetime(
    df["record_generated_date"],
    errors="coerce"
)

df["enrollment_date"] = pd.to_datetime(
    df["enrollment_date"],
    errors="coerce"
)


In [ ]:
print(df[["record_generated_date", "enrollment_date"]].dtypes)

In [ ]:
print(
    df[["record_generated_date", "enrollment_date"]].isnull().sum()
)


In [ ]:
pass  # duplicate income-log step removed; handled above


In [ ]:
pass  # duplicate income-log step removed; handled above


In [ ]:
df["household_income_log"]

In [ ]:
print(df[[
    "household_income_monthly_usd",
    "household_income_log"
]].head())


In [ ]:
print(df["household_income_log"].describe())

In [ ]:
print(df["dropout_risk_level"].value_counts())

In [ ]:
print(df["dropout_probability_score"].describe())

In [ ]:
print(
    df.groupby("dropout_risk_level")[
        "dropout_probability_score"
    ].agg(["count", "min", "mean", "max"])
)


In [ ]:
df["record_generated_date"] = pd.to_datetime(
    df["record_generated_date"],
    errors="coerce"
)

df["enrollment_date"] = pd.to_datetime(
    df["enrollment_date"],
    errors="coerce"
)

print(
    df[["record_generated_date", "enrollment_date"]].isnull().sum()
)


In [ ]:
print(df["dropout_risk_level"].value_counts())

In [ ]:
print(df["dropout_probability_score"].describe())

In [ ]:
print(
    df.groupby("dropout_risk_level")[
        "dropout_probability_score"
    ].agg(["count", "min", "mean", "max"])
)


In [ ]:
y = df["dropout_risk_level"]

In [ ]:
model_df = df.drop(
    columns=[
        "student_id",
        "dropout_probability_score",
        "dropout_risk_level",
        "case_notes"
    ]
).copy()


In [ ]:
print(model_df.shape)

In [ ]:
print(model_df.dtypes)

In [ ]:
if {"enrollment_date", "record_generated_date"}.issubset(model_df.columns):
    print(
        "Enrollment after record date:",
        (model_df["enrollment_date"] > model_df["record_generated_date"]).sum(),
    )
else:
    print("Date columns already converted/dropped.")


In [ ]:
print(
    "Minimum enrollment duration:",
    (
        model_df["record_generated_date"]
        - model_df["enrollment_date"]
    ).dt.days.min()
)

print(
    "Maximum enrollment duration:",
    (
        model_df["record_generated_date"]
        - model_df["enrollment_date"]
    ).dt.days.max()
)


In [ ]:
model_df["record_year"] = model_df["record_generated_date"].dt.year
model_df["record_month"] = model_df["record_generated_date"].dt.month

model_df["enrollment_year"] = model_df["enrollment_date"].dt.year
model_df["enrollment_month"] = model_df["enrollment_date"].dt.month

model_df["enrollment_duration_days"] = (
    model_df["record_generated_date"]
    - model_df["enrollment_date"]
).dt.days


In [ ]:
cols_to_drop = [c for c in ["record_generated_date", "enrollment_date"] if c in model_df.columns]
if cols_to_drop:
    model_df.drop(columns=cols_to_drop, inplace=True)
    print("Dropped:", cols_to_drop)
else:
    print("Date columns already dropped.")


In [ ]:
cols_to_drop = [c for c in ["record_generated_date", "enrollment_date"] if c in model_df.columns]
if cols_to_drop:
    model_df.drop(columns=cols_to_drop, inplace=True)
    print("Dropped:", cols_to_drop)
else:
    print("Date columns already dropped.")


In [ ]:
y = df["dropout_risk_level"]

In [ ]:
if {"enrollment_date", "record_generated_date"}.issubset(model_df.columns):
    print(
        "Enrollment after record date:",
        (model_df["enrollment_date"] > model_df["record_generated_date"]).sum(),
    )
else:
    print("Date columns already converted/dropped.")


In [ ]:
cols_to_drop = [c for c in ["record_generated_date", "enrollment_date"] if c in model_df.columns]
if cols_to_drop:
    model_df.drop(columns=cols_to_drop, inplace=True)
    print("Dropped:", cols_to_drop)
else:
    print("Date columns already dropped.")


In [ ]:
print("record_generated_date" in model_df.columns)
print("enrollment_date" in model_df.columns)


In [ ]:
print(model_df[
    [
        "record_year",
        "record_month",
        "enrollment_year",
        "enrollment_month",
        "enrollment_duration_days"
    ]
].head())


In [ ]:
print(model_df.shape)
print(model_df.columns.tolist())


In [ ]:
print(model_df["academic_year"].value_counts().sort_index())

In [ ]:
print(
    model_df[
        [
            "academic_year",
            "record_year",
            "enrollment_year"
        ]
    ].head(20)
)


In [ ]:
model_df["academic_year_start"] = (
    model_df["academic_year"]
    .str.split("-")
    .str[0]
    .astype(int)
)


In [ ]:
model_df["academic_year_end"] = (
    model_df["academic_year"]
    .str.split("-")
    .str[1]
    .astype(int)
)


In [ ]:
print(
    model_df[
        [
            "academic_year",
            "academic_year_start",
            "academic_year_end"
        ]
    ].head(20)
)


In [ ]:
print(
    model_df["academic_year_start"].value_counts().sort_index()
)


In [ ]:
print(
    model_df["academic_year_end"].value_counts().sort_index()
)


In [ ]:
print(
    model_df["academic_year_end"].value_counts().sort_index()
)


In [ ]:
model_df.drop(
    columns=["academic_year"],
    inplace=True
)


In [ ]:
model_df.drop(
    columns=["academic_year_end"],
    inplace=True
)


In [ ]:
print(df["school_infrastructure_score"].isnull().sum())

In [ ]:
print("Median:",
      df["school_infrastructure_score"].median())


In [ ]:
print(df["school_infrastructure_score"].describe())

In [ ]:
numerical_imputed_cols = [
    "distance_to_school_km",
    "household_income_monthly_usd",
    "family_size",
    "attendance_rate_pct",
    "average_test_score_pct",
    "teacher_student_ratio",
    "school_infrastructure_score"
]

print(df[numerical_imputed_cols].isnull().sum())


In [ ]:
#feature engineering

In [ ]:
# Features
X = model_df.copy()

# Target
y = df["dropout_risk_level"].copy()


In [ ]:
if "student_id" in X.columns:
    X = X.drop(columns=["student_id"])


In [ ]:
if "dropout_probability_score" in X.columns:
    X = X.drop(columns=["dropout_probability_score"])


In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX columns:")
print(X.columns.tolist())

print("\nTarget:")
print(y.value_counts())


In [ ]:
print("Target in X:", "dropout_risk_level" in X.columns)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


## Model training (XGBoost)

Train a multiclass dropout-risk model on the engineered features using the same train/test split from above.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from xgboost import XGBClassifier
import joblib

# Normalize feature types for sklearn
for frame in (X_train, X_test):
    for col in frame.columns:
        if frame[col].dtype == "bool":
            frame[col] = frame[col].astype(int)
        elif frame[col].dtype == "object":
            frame[col] = frame[col].astype(str)

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train.astype(str))
y_test_enc = label_encoder.transform(y_test.astype(str))

numeric_cols = numeric_only_columns(X_train)
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_cols),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_cols,
        ),
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        eval_metric="mlogloss",
        random_state=42,
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
    )),
])

model.fit(X_train, y_train_enc)
print("Model training complete.")

In [ ]:
raw_pred = model.predict(X_test)
if getattr(raw_pred, "ndim", 1) > 1:
    raw_pred = raw_pred.argmax(axis=1)
y_pred = label_encoder.inverse_transform(raw_pred.astype(int))
y_true = label_encoder.inverse_transform(y_test_enc)

print(classification_report(y_true, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred, labels=label_encoder.classes_))


## SHAP explainability


In [ ]:
import shap

sample = X_test.sample(min(300, len(X_test)), random_state=42)
transformed = model.named_steps["preprocessor"].transform(sample)
explainer = shap.TreeExplainer(model.named_steps["classifier"])
shap_values = explainer.shap_values(transformed)
print("SHAP values computed for", len(sample), "students")


In [ ]:
from pathlib import Path

model_path = MODELS_DIR / "education_risk_notebook_model.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": model, "label_encoder": label_encoder}, model_path)
print(f"Saved model to {model_path.resolve()}")
